# Computational Theory Assessment

This notebook contains my solutions to the problems in the Computational Theory module.
The problems are based on the SHA-256 hash algorithm, as specified in [FIPS 180-4: Secure Hash Standard (NIST)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

Throughout the notebook references such as *Section 3.2* are referencing the sections in that document.

Each problem has its own section. The section will state the problem, explains the approach taken and
then works through the solution in small code cells.

## Problem 1: Representing SHA-256 Data

### The Problem


> Investigate how the input, output and intermediate data can be represented in Python.
> Explain how you would represent:
>
> - 32-bit words
> - sequences of 32 bit words
> - input messages
> - 512 bit message blocks
> - the final 256 bit hash value
> - big endian integers

### 1. 32-bit Words

#### What the standard says

[Secure Hash Standard - FIPS PUB 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), *Section 2.1*, defines a word as:

> "A group of either 32 bits (4 bytes) or 64 bits (8 bytes), depending on the secure hash algorithm."

SHA-1, SHA-224 and SHA-256 use 32-bit (4-byte) words. SHA-384 and SHA-512 use 64-bit (8-byte) words.
*Section 3.2* confirms this: "SHA-1, SHA-224 and SHA-256 operate on 32-bit words (w=32)".

*Section 3.2* also defines how words are added. Two words are treated as integers $X$ and $Y$ and the result is
$Z = (X + Y) \bmod 2^{32}$. Any carry out of the top bit is thrown away, so the answer always fits back into 32 bits.

A representation of a 32-bit word therefore needs to:

- hold any value from $0$ to $2^{32} - 1$,
- wrap around modulo $2^{32}$ on addition, and
- keep bitwise operations within 32 bits.

#### Python's built in `int`

[The Python docs on Numeric Types](https://docs.python.org/3/builtins/stdtypes.html#numeric-types-int-float-complex) state that "integers have unlimited precision". An `int` can store any 32 bit value, but it has no fixed width. It never wraps. it just keeps growing.

The cell below shows this with the largest 32 bit value.

In [42]:
import math

# The largest value that fits in 32 bits: 2**32 - 1, which is 32 one bits.
MAX_WORD = 0xFFFFFFFF

# show the value and how many bits it needs
print(f"{MAX_WORD=} bits needed: {MAX_WORD.bit_length()}")

# Add 1. A 32 bit word should wrap to 0, but a python int does not
x = MAX_WORD + 1

# The result needs 33 bits, so it is no longer a valid word.
print(f"{x=} bits needed: {x.bit_length()}")

MAX_WORD=4294967295 bits needed: 32
x=4294967296 bits needed: 33


#### Option A: `int` with a mask

One option is to keep using `int` and cut the result back to 32 bits after every
operation with a [bitwise AND](https://docs.python.org/3/library/stdtypes.html#bitwise-operations-on-integer-types)

`x & 0xFFFFFFFF` keeps only the lowest 32 bits, which is the same as $x \bmod 2^{32}$

In [43]:
# Addition, then mask back to 32 bits. This matches Section 3.2.
print(f"{(MAX_WORD + 1) & MAX_WORD = }")

# Left shifts also grow an int beyond 32 bits.
print(f"{hex(MAX_WORD << 4) = }")

# Masking the shifted value brings it back to   32 bits
print(f"{hex((MAX_WORD << 4) & MAX_WORD) }")

(MAX_WORD + 1) & MAX_WORD = 0
hex(MAX_WORD << 4) = '0xffffffff0'
0xfffffff0


This works. The weakness is that the width is not part of the type: if a mask is forgotten, the code still runs but produces the wrong hash, with no error.

The shift above shows why the mask matters. *Section 2.2.2* defines `x << n` as "discarding the left-most n bits of the word x and then padding the result with n zeroes on the right". A plain `int` keeps those bits (`0xffffffff0`), so it only matches the standard once it is masked (`0xfffffff0`).

#### Option B: `numpy.uint32`

NumPy provides [fixed-width integer types](https://numpy.org/doc/stable/user/basics.types.html)
`numpy.uint32` is a unsigned integer that is always exactly 32 bits wide



In [44]:
# NumPy provides fixed-width integer types such as uint32.
import numpy as np

In [45]:
# Create 32 bit unsigned word holding the largest value
w = np.uint32(MAX_WORD)

# show its type and its size in bytes
print(f"{type(w)=}, {w.itemsize=} bytes")

type(w)=<class 'numpy.uint32'>, w.itemsize=4 bytes


A **scalar** is a single value, unlike an array, which can hold many values. For example, `np.uint32(5)` is a
NumPy scalar: one 32-bit unsigned integer. `np.array([5, 6], dtype=np.uint32)` is an array of two of them.
NumPy scalars have the same attributes and methods as arrays, but they are immutable
([NumPy docs: Scalars](https://numpy.org/doc/stable/reference/arrays.scalars.html)).

In [46]:
# A scalar is a single value.  it has no dimensions.
print(type(w), np.ndim(w))

# An array holds values along one or more dimensions.
a = np.array([5, 6], dtype=np.uint32)
print(type(a), np.ndim(a))

<class 'numpy.uint32'> 0
<class 'numpy.ndarray'> 1


Adding to a `uint32` wraps modulo $2^{32}$ as section 3.2 requires. For single values. NumPy will throw a `RuntimeWarning`, is most programs this is a mistake.

[NumPy docs: Overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)

In [47]:
# scalar addition: wraps to 0, and NumPy warns about the overflow
w + np.uint32(1)
np.uint32(0)

/var/folders/25/4phy0jxx3ln9rwmnr12hc7g80000gn/T/ipykernel_8678/445645005.py:2: RuntimeWarning: overflow encountered in scalar add
  w + np.uint32(1)


np.uint32(0)

In SHA-256 the wrap is intended. so the warning is noise. The
[`numpy/errstate`](https://numpy.org/doc/stable/reference/generated/numpy.errstate.html)
context manager will tell NumPy to ignore overflow inside a block of code.
    Its documentation describes it as handling floating-point errors,
but it will also silence the integer scalar overflow warning.

In [48]:
# let NumPy know the overflow is intended. so it will not throw more warnings.
with np.errstate(over='ignore'):
    # Wraps tp 0, silently
    print(repr(w + np.uint32(1)))

np.uint32(0)


In [49]:
# Arithmetic on arrays of uint32 wraps silently, with no warning.
np.array([MAX_WORD], dtype=np.uint32) + np.uint32(1)

array([0], dtype=uint32)

In [50]:
# a left shift stays within 32 bits, in section 2.2.2 it defines that the top 4 bits will fall off the end
hex(w << np.uint32(4))

'0xfffffff0'

NumPy also will not create a word from a value that does not fit, instead of silently truncating it


In [51]:
# 2**32 needs 33 bits. so it cant be stored in a uint32

try:
    np.uint32(2**32)
except OverflowError as e:
    # show the error
    print(f"OverflowError: {e}")

OverflowError: Python integer 4294967296 out of bounds for uint32


One issue, if you mix a `uint32` with a python `float` it produces a `float64`
this would break the bitwise operations SHA-256 needs. Constants should be written as integers

In [52]:
# A small word, so the additions below do not overflow.
small = np.uint32(5)

# Adding an int keeps the uint32 type.
print(type(small + 1))

# Adding a float silently changes the type to float64.
print(type(small + 1.0))

<class 'numpy.uint32'>
<class 'numpy.float64'>


#### Decision
We represent 32 bit words with **numpy.uint32** The 32 bit width is part of the type, so wrapping modulo $2^{32}$ (Section 3.2) happens automatically and out of range values raise an error instead of being silently accepted.
The costs are the scalar overflow warnings, which we suppress with `np.errstate` where the wrap is intended
and the need to avoid floats. A plain `int` with `& 0xFFFFFFFF` mask would be right. but it relies on never forgetting the mask

### 2. Sequences of 32 bit words

#### What the standard says

*Section 6.2* says SHA-256 may be used to hash a message.
The algorithm uses:
1. "a message schedule of sixty four 32 bit words,"
2. "eight working variables of 32 bits each"
3. "a hash value of eight 32 bit words". 

The final result of SHA-256 is a 256-bit message digest

#### Options

- A python **`list`** of `int`s. it is flexible. but the elements can be anything, including values that are too big.
- A numpy **`array`** with `dtype=np.uint32`
  [NumPy docs: array creation](https://numpy.org/doc/stable/user/basics.creation.html)
-> Every element has the same fixed 32 bit type, arithmetic applies to every element at once
  
  


In [53]:
# The eight start hash words for SHA-256, copied from FIPS 180-4, Section 5.3.3

H0 = np.array([
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19
], dtype=np.uint32)

print([hex(h) for h in H0])

['0x6a09e667', '0xbb67ae85', '0x3c6ef372', '0xa54ff53a', '0x510e527f', '0x9b05688c', '0x1f83d9ab', '0x5be0cd19']


[`ndarray.shape`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html) gives the number of
elements along each dimension, and [`ndarray.nbytes`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html)
gives the total bytes used by the elements.

In [54]:
# Number of words and the type of each element.
print(f"{H0.shape=}, {H0.dtype=}")

# Each word is 4 bytes, so 8 words take 32 bytes, which is 256 bits.
print(f"{H0.itemsize=} bytes, {H0.nbytes=} bytes, {H0.nbytes * 8} bits")

H0.shape=(8,), H0.dtype=dtype('uint32')
H0.itemsize=4 bytes, H0.nbytes=32 bytes, 256 bits


A`list` accepts values that are not valid words. A `uint32` array rejects them.

In [55]:
# A list happily stores a 41bit number next to a valid word. 
bad = [0xFFFFFFFF,  ]
print(f"list: {bad}")

# The same values in a uint32 array raise an error
try:
    np.array(bad, dtype=np.uint32)
except OverflowError as e:
    print(f"array: OverflowError: {e}")

list: [4294967295]


At the end of each block, *Section 6.2.2* adds the eight working variables to the eight hash words.

In [56]:
# Addition of two 8 word sequences. Each element wraps modulo 2**
print([hex(h) for h in H0 + H0])

['0xd413ccce', '0x76cf5d0a', '0x78dde6e4', '0x4a9fea74', '0xa21ca4fe', '0x360ad118', '0x3f07b356', '0xb7c19a32']


In [57]:
# The 64 word message schedule is created and filled with zeros.
w = np.zeros(64, dtype=np.uint32)

# 64 words of 4 bytes each is 256 bytes
print(f"{w.shape=}, {w.nbytes=} bytes")

w.shape=(64,), w.nbytes=256 bytes


#### Decision

If you represent equences of words as **Numpy arrays with `dtype=np.uint32`**. Every element is guaranteed to be a valid 32 bit word

### 3. Input Messages

#### What the standard says

Figure 1 in Section 1 of [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf) gives the
SHA-256 message size as less than $2^{64}$ bits.

*Section 2.2.1* lists $\ell$ as the "Length of the message, M, in bits". *Section 5.1.1*, which describes
padding, gives an example: "the (8-bit ASCII) message "abc" has length 8×3 = 24", and its diagram shows
"abc" as the bits `01100001 01100010 01100011` with $\ell = 24$. The length $\ell$ is used later in padding.

![FIPS 180-4 Section 5.1.1: padding the message "abc" to 512 bits](img/fips-180-4-section-5-1-1.jpg)

*Section 5.1.1 of [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), page 13.*

#### Options

Python [`bytes`](https://docs.python.org/3/builtins/stdtypes.html#bytes-objects) type us an immutable sequence of integers from 0 to 255, one per byte. 
This is a fit and is what `hashlib` accepts.

There is one limitation. `bytes` can only hold a whole number of bytes. So it can't represent messages whose length is not divisible by 8.

The standard allows such messages, but [`hashlib`](https://docs.python.org/3/library/hashlib.html) does not support them either. 

In [58]:
# Test must be encoded to bytes before hashing. 'abc' is the example used by NIST.
message = "abc".encode("utf-8")

# Show the bytes object, its type and the integer value of each byte.
print(message, type(message), list(message))

b'abc' <class 'bytes'> [97, 98, 99]


In [59]:
# The message length in bits which is the 'l' value in FIPS 180-4
length_bits = len(message) * 8

# Section 5.1.1 gives 24 for "abc"
print(f"{length_bits=}")

length_bits=24


A python "str" is a sequence of characters, not bytes. It must be encoded "[see docs on unicode](https://docs.python.org/3/howto/unicode.html)" before hashing.
In UTF-8 one character can take between one and four bytes, so the number of characters is not always the number of bytes

#### Descision

We represent input messages as **`byte`**, with text encoded as **UTF-8** and use a **`bytearray`** while building the padded message. The length $\ell$ in bits is `len(message) * 8`. We only support
messages that are a whole number of bytes, which matches `hashlib`.

### 4. 512-bit Message Blocks

#### What the standard says 

After padding (Section 5.1.1) the message length is a multiple of 512 bits.

Section 5.2.1 then parses it into $N$ blocks $M^{(1)}, \dots, M^{(N)}$, where each block is sixteen 32-bit words $M_0^{(i)}, \dots, M_{15}^{(i)}$.

*Section 3.1* (item 4a) says the same: each 512-bit block is "represented as a sequence of sixteen 32-bit words".

$512$ bits $= 64$ bytes $= 16$ words of $4$ bytes.


#### Options

A 512-bit block can be held in two ways:

- **64 `bytes`**: easy to cut out of the message with slicing.
- **16 `uint32` words**: what the hash computation works on. Step 1 of *Section 6.2.2* copies the
  16 words of the block, $M_0^{(i)}$ to $M_{15}^{(i)}$, into the first 16 words of the message schedule.

using both: slice the message into 64byte blocks, then turn each block into 16 words.

In [60]:
# A block is 512 bits, which is 64 bytes
BLOCK_SIZE = 64

# Test data standing in for a padded message: the bytes 0,1,2...127
data = bytes(range(128))

# its length is a multiple of 512 bits, as it should be with padding
print(f"{len(data)} bytes = {len(data) * 8} bits")

128 bytes = 1024 bits


In [61]:
# Slice the data into 64-byte blocks.
blocks = [data[i:i + BLOCK_SIZE] for i in range(0, len(data), BLOCK_SIZE)]

# Show how many blocks there are and the size of each.
print(f"{len(blocks)} blocks, sizes: {[len(b) for b in blocks]}")

2 blocks, sizes: [64, 64]


The list comprehension above uses [range](https://docs.python.org/3/library/stdtypes.html#range) with a step of 64 to get the start of each block, and a [slice](https://docs.python.org/3/library/stdtypes.html#common-sequence-operations) to take the 64 bytes that follow

To turn a block into words we can use a [`numpy.frombuffer`](https://numpy.org/devdocs/reference/generated/numpy.frombuffer.html) - this will read the bytes directly as an array.

The dtype `">u4"` means big-endian (`>`), unsigned (`u`), 4 bytes (`4`

In [62]:
# Read the first block as 16 big endian 32 bit words. 
M = np.frombuffer(blocks[0], dtype=">u4").astype(np.uint32)

# Show the shape, the type, and the first four words. 
print(f"{M.shape=}, {M.dtype=}")
print([hex(m) for m in M[:4]])

M.shape=(16,), M.dtype=dtype('uint32')
['0x10203', '0x4050607', '0x8090a0b', '0xc0d0e0f']


In [63]:
# The first word is the first four bytes read as a big endian integer.
print(M[0] == int.from_bytes(blocks[0][:4], "big"))

True


In [64]:
def block_to_words(block):
    # Convert a 64 byte (512 bit) block into 16 big-endian 32-bit words - FIPS 180-4, Section 5.2.1)
    # A block must be exactly 64 bytes
    if len(block) != BLOCK_SIZE:
        raise ValueError(f"Block must be {BLOCK_SIZE} bytes, it is {len(block)}")
        # read the big endian words and convert to native uint32
    return np.frombuffer(block,dtype=">u4").astype(np.uint32)

In [65]:
# Convert every block and show the shape of each result.
print([block_to_words(b).shape for b in blocks])

[(16,), (16,)]


#### Decision 

Split the padded message into **64byte slices** and convert each one into a **NumPy array of 16 `uint32` words with `block_to_words`
, reading the bytes as big endian as the standard says.

### 5. The final 256 bit Hash Value

#### What the standard says 

At the end of Section 6.2.2, after all $N$ blocks are processed, the message digest is the eight hash words joined together:

$$H_0^{(N)} \| \dots \| H_7^{(N)}$$

$8$ words $\times$ $32$ bits $= 256$ bits $= 32$ bytes

*Section 2.2.1* defines $H^{(N)}$ as "the final hash value and is used to determine the message digest".
*Section 5.3.3* only gives the initial value $H^{(0)}$.

#### Options

The standard doc keeps the hash value as eight 32 bit words. and the final digest is those words joined. That one 256 bit value can be held in three forms

- **8 `uint32` words**: how *Section 6.2.2* holds it during the computation.
- **32 `bytes`**: the 8 words written out in big endian order (*Section 3.1*), 8 × 4 = 32 bytes.
  This is what [`hashlib`](https://docs.python.org/3/library/hashlib.html)'s `.digest()` returns.
- **a 64-character hex string**: *Section 3.1* writes words as hex digits, one per 4 bits, so 256 bits
  is 64 hex digits. NIST's [SHA-256 example](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf)
  gives the digest of "abc" in this form, and `hashlib`'s `.hexdigest()` returns it.


In [66]:
import hashlib

# the sha 256 digest of "abc" as raw bytes
digest = hashlib.sha256(b"abc").digest()

# 32 bytes which is 256 bits.
print(f"{len(digest)} bytes = {len(digest) * 8} bits")

32 bytes = 256 bits


In [67]:
# the same digest as a hex string: two hex characters per bytem so 64 characters.
print(digest.hex(), len(digest.hex()))

ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad 64


This matches the expected digest for "abc" in [NIST SHA 256 worked example](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf)

In [68]:
# Bytes to words: read the 32 bytes as 7 big endian 32 bit words
H = np.frombuffer(digest, dtype=">u4").astype(np.uint32)

# show the eight words
print([hex(h) for h in H])

['0xba7816bf', '0x8f01cfea', '0x414140de', '0x5dae2223', '0xb00361a3', '0x96177a9c', '0xb410ff61', '0xf20015ad']


You can turn words back to bytes using numpy again

[`ndarray.tobytes`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.tobytes.html)
writes the array in its own byte order. If you use a `uint32` array it handles it in reverse.

In [69]:
# uint32 byte order. because regular compute is weighted towards little endian that is Intel, AMD, mac 
print(H.tobytes().hex())
# Correct :  convert to big endian first, then write the bytes.
print(H.astype(">u4").tobytes().hex())

bf1678baeacf018fde4041412322ae5da36103b09c7a179661ff10b4ad1500f2
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


In [70]:
def words_to_hex(words):
    # Join 8 32-bit words into the 64-character hex digest (FIPS 180-4, Section 6.2.2)
    # Write each word as 4 big endian bytes, then convert all 32 bytes to hex.
    return words.astype(">u4").tobytes().hex()

In [71]:
# words back to hex gives the same digest
print(words_to_hex(H) == hashlib.sha256(b"abc").hexdigest())

True


### Decision



int.from_bytes
int.to_bytes

Big endian -> most important byte comes first
SHA-256 uses this order everywhere. 

Python had three ways to turn byte into a number and back

- `int.from_bytes`
- `int/to_bytes`
#### Decision

We use `int.from_bytes` and `int.to_bytes` with `"big"` for single numbers like the message ength, and the NumPy dtype `">u4"` for whole blocks and the final hash. `struct` gives the same esults,but we don't need it for single values. The byte order is always written out, so the result s the same on any computer.

### Summary

We represent each 32 bit word as a NumPy uint32, because it is always exactly 32 bits and wraps around on its own

## Imports

- [`hashlib`](https://docs.python.org/3/library/hashlib.html): Python's built in SHA256
- [`struct`](https://docs.python.org/3/library/struct.html): packs values with an explicit byte order
- [`sys`](https://docs.python.org/3/library/sys.html#sys.byteorder): reports this machines byte order
- [`numpy`](https://numpy.org/doc/stable/): fixed width integer types and arrays

In [72]:
# I think it is best to leave the imports at the end and cleanup the rest in the problem

# SHA-256 from the standard library used to check results
import hashlib

# Packing and unpacking binary data with an explicit byte order
import struct

# System information, including the native byte order
import sys

# Numerical arrays with fixed width integer types such as uint32
import numpy as np

## Problem 2: SHA-256 Bitwise Operations

### Implement the Boolean and bitwise functions used by SHA-256:

- Ch(x, y, z)
- Maj(x, y, z)
- `Sigma0(x)`: written as $\Sigma_0^{\{256\}}(x)$ in the standard
- `Sigma1(x)`: written as $\Sigma_1^{\{256\}}(x)$ in the standard
- `sigma0(x)`: written as $\sigma_0^{\{256\}}(x)$ in the standard
- `sigma1(x)`: written as $\sigma_1^{\{256\}}(x)$ in the standard

* Use NumPy so that the values involved in these operations are represented as 32-bit unsigned integers.

For each function:

> - provide an appropriate docstring
> - explain the operation in Markdown
> - implement the function directly from that definition
> - provide tests on appropriate inputs

SHA-256 uses six small functions to mix up the bits of its 32-bit words, and we implement each one as a short
Python function using `numpy.uint32` values, written straight from the equations in *Section 4.1.2*

`Ch(x, y, z)` is "choose": for each bit, $x$ decides whether the answer comes from $y$ (where $x$ is 1) or from
$z$ (where $x$ is 0)

`Maj(x, y, z)` is "majority": each bit of the answer is whatever at least two of $x$, $y$ and $z$ agree on
Both are built only from AND, XOR and NOT, applied to whole words at once

The other four each take one word and XOR together three rotated or shifted copies of it:

- `Sigma0` rotates by 2, 13 and 22.
- `Sigma1` rotates by 6, 11 and 25.
- `sigma0` rotates by 7 and 18, then shifts by 3.
- `sigma1` rotates by 17 and 19, then shifts by 10.

Because a rotation never loses bits, the uppercase functions mix the word without throwing anything away
The shift drops bits off the end, which is the key difference between the upper-case and lower-case functions

Python has no rotate operator, so we build rotation from two shifts and an OR, as *Section 3.2* defines it
In SHA-256, the lowercase sigmas build the message schedule, and the uppercase sigmas, `Ch` and `Maj` are
used in every one of the 64 rounds (*Section 6.2.2*)

### What the standard says

*Section 3.2* defines two operations on 32 bit words:

- $SHR^n(x)$: shift right by $n$. Bits that fall off are lost
- $ROTR^n(x)$: rotate right by $n$. Bits that fall off come back on the left

Python has no rotate, so `rotr` shifts right and  shifts left by $32 - n$, and ORs the two together

In [73]:
def shr(x, n):
    """Shift the 32bit word x right by n bits: SHR^n(x), FIPS 180-4 Section 3.2."""
    return np.uint32(x) >> np.uint32(n)


def rotr(x, n):
    """Rotate the 32-bit word x right by n bits, 0 < n < 32: ROTR^n(x), FIPS 180-4 Section 3.2."""
    x = np.uint32(x)
    return (x >> np.uint32(n)) | (x << np.uint32(32 - n))

In [74]:
def Ch(x, y, z):
    """Choose: where x is 1 take y bit, where x is 0 take z bit. FIPS 180-4, equation 4.2."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (~x & z)


def Maj(x, y, z):
    """Majority: each bit is whatever at least two of x, y and z agree on. FIPS 180-4, equation 4.3."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (x & z) ^ (y & z)


def Sigma0(x):
    """ROTR^2 XOR ROTR^13 XOR ROTR^22. FIPS 180-4, equation 4.4."""
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)


def Sigma1(x):
    """ROTR^6 XOR ROTR^11 XOR ROTR^25. FIPS 180-4, equation 4.5."""
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)


def sigma0(x):
    """ROTR^7 XOR ROTR^18 XOR SHR^3. FIPS 180-4, equation 4.6."""
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)


def sigma1(x):
    """ROTR^17 XOR ROTR^19 XOR SHR^10. FIPS 180-4, equation 4.7."""
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

#### Decision

Each function is a short Python function on `numpy.uint32`, built from `shr` and `rotr` The tests show the
upper case sigmas keep every bit, while the shifts in the lower case sigmas throw bits away

#### References

This video by [Xiuminseokie21 on YouTube](https://youtu.be/9xs4eWOAG7Y?si=DK2XBgbIyxM9DzVN) has a great implementation of these functions, with a visual.

## Problem 3: SHE-256 Round Constants

### The Problem

The Secure Hash Standard defines the initial hash values and round constants used by SHA-256 using fractional parts of square and cube roots of prime numbers.

* Use Python and NumPy to generate the 64 round constants specified in the standard.

Your solution should:

> - write a function primes(n) that generates the first n prime numbers,
> - calculate the cube roots of the first 64 primes,
> - extract the fractional part of each cube root,
> - convert the resulting to a 32-bit unsigned integer,
> - display the resulting constants in hexadecimal,
> - compare your results with the constants in the Secure Hash Standard.

## Problem 3: Generating the SHA-256 Constants

### The Problem

> The Secure Hash Standard defines the initial hash values and round constants used by SHA-256 using fractional
> parts of square and cube roots of prime numbers.

> Use Python and NumPy to generate the 64 round constants specified in the standard.

> Your solution should:

> 1. write a function `primes(n)` that generates the first `n` prime numbers
> 2. calculate the cube roots of the first 64 primes
> 3. extract the fractional part of each cube root
> 4. convert the resulting to a 32 bit unsigned integer
> 5. display the resulting constants in hexadecimal
> 6. compare your results with the constants in the Secure Hash Standard

### What the standard says

*Section 4.2.2* of [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf) says the 64 round
constants are "the first 32  bits of the fractional parts of the cube roots of the first 64 prime
numbers". The initial hash values from Problem 1 are made the same way, using square roots of the first eight
primes (*Section 5.3.3*).

### Step 1: `primes(n)`

This builds on the module notes
([Prime Numbers](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/prime-numbers.ipynb)).
`isprime_sqrt` is taken from the notes: it tests a number by trying every divisor up to its square root.
`primes(n)` adapts the notes' `find_primes()`, which finds the first ten primes, so that it finds the first `n` instead.

In [77]:
# From the module notes: test whether n is prime by trying divisors up to its square root.
def isprime_sqrt(n):
    """Test whether n is prime."""
    for i in range(2, np.floor(np.sqrt(n)).astype(int) + 1):
        if n % i == 0:
            return False
    return True


# Adapted from find_primes() in the module notes, to return the first n primes instead of ten.
def primes(n):
    """Return the first n prime numbers."""
    found = []
    candidate = 2
    while len(found) < n:
        if isprime_sqrt(candidate):
            found.append(candidate)
        candidate += 1
    return found

In [78]:
# The first 8 primes, the count and last of the first 64
print(primes(8))
print(len(primes(64)), primes(64)[-1])

[2, 3, 5, 7, 11, 13, 17, 19]
64 311


#### Why this works

- **Why stop at the square root?** If $n$ is not prime, it can be written as $n = a \times b$. Both $a$ and $b$
  can't be bigger than $\sqrt{n}$, because then $a \times b$ would be bigger than $n$. So if $n$ has a divisor,
  at least one is no bigger than $\sqrt{n}$, and checking up to there is enough.
- **What `np.floor(np.sqrt(n)).astype(int) + 1` does.** It takes the square root, rounds it down to a whole
  number, and adds 1 because `range` stops one before its end value. For $n = 97$: $\sqrt{97} \approx 9.85$,
  rounded down to 9, so `range(2, 10)` tries 2 to 9.
- **How `primes(n)` uses it.** It starts at 2 and tests each number in turn, keeping the ones that are prime,
  until it has `n` of them.

### Step 2-4:

In [ ]:

# Step 2: cube root of each of the first 64 primes.
roots = np.cbrt(primes(64))

# Step 3: keep only the part after the decimal point.
fractions = roots % 1

# Step 4: multiply by 2**32 to move the first 32 bits in front of the point, then keep them as uint32.
constants = (fractions * 2**32).astype(np.uint32)

#### Why this works

- **`% 1` gives the fractional part.** The remainder after dividing by 1 is whatever is after the decimal point,
  so $1.2599\ldots \bmod 1 = 0.2599\ldots$
- **Multiplying by $2^{32}$ gives the first 32 bits.** Multiplying by 2 moves a binary fraction one place left,
  so multiplying by $2^{32}$ moves its first 32 bits in front of the point. Converting to `uint32` drops
  everything after the point and keeps exactly those 32 bits.
- **Floats are precise enough.** A 64-bit float holds 53 significant bits. The largest cube root,
  $\sqrt[3]{311} \approx 6.8$, needs 3 bits before the point, plus the 32 we keep, which is 35 bits, well
  within 53. The comparison with the standard confirms all 64 come out right.